# Prefect — ETL de control contractual

Reinicie el kernel si se modificaron los módulos Python antes de ejecutar. Ejecute primero la celda que configura la ruta del proyecto.

La celda principal permite elegir una sola operación. Queda seleccionada **solo_postgres** para recuperar la carga fallida utilizando Silver y Gold existentes. Este modo no actualiza los datos desde la fuente ni recalcula Gold.


In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'secop_pipeline').exists():
    ROOT = ROOT.parent
if not (ROOT / 'secop_pipeline').exists():
    raise RuntimeError('Abra el notebook desde la carpeta del proyecto o notebooks')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
# ELIJA UN SOLO MODO antes de ejecutar esta celda.
# Reinicie el kernel si se actualizó load.py u otro módulo del proyecto.
# No lance otra ejecución mientras haya un ETL activo.

MODO ="reprocesar_local"  # Recuperación actual: Silver y Gold ya están generados.

# "solo_postgres": carga los archivos Silver y Gold existentes en PostgreSQL.
#   No descarga, no transforma y no recalcula indicadores.
#   Úselo cuando Gold terminó correctamente y falló únicamente la carga.
#   Si los archivos no cambian, volverá a cargar la misma información.
#   Es una carga directa: no crea un nuevo informe completo de Prefect
#   ni cambia el estado FALLIDA del informe de la ejecución anterior.
#
# "reprocesar_local": ejecuta el flujo con refresh=False.
#   Reutiliza Bronze, reutiliza Silver si su caché sigue vigente o lo transforma,
#   ejecuta calidad, RECONSTRUYE Gold y carga PostgreSQL.
#   Úselo para aplicar cambios de lógica a los datos ya descargados.
#   No incorpora datos nuevos de la API.
#
# "actualizar_todo": ejecuta el flujo con refresh=True.
#   Descarga nuevamente las cuatro bases; transforma o reutiliza Silver
#   según la caché, ejecuta calidad, reconstruye Gold y carga PostgreSQL.
#   Úselo para actualizar los datos desde la fuente. Es la opción más larga.
#   Revise las fechas configuradas y los parámetros de ircd_config.json.
#
# replace=True reemplaza las tablas al finalizar correctamente la carga.
# No agrega filas a las tablas anteriores; la carga es transaccional.

if MODO == "solo_postgres":
    from secop_pipeline.load import load_all_to_postgres
    resultado = load_all_to_postgres(replace=True)
elif MODO in ("reprocesar_local", "actualizar_todo"):
    from flows.secop_flow import secop_flow
    resultado = secop_flow(
        refresh=(MODO == "actualizar_todo"),
        load_postgres=True,
    )
else:
    raise ValueError(
        "MODO debe ser solo_postgres, reprocesar_local o actualizar_todo"
    )

resultado


08:57:41.428 | INFO    | Flow run 'mellow-cuttlefish' - Beginning flow run 'mellow-cuttlefish' for flow 'SECOP-II-CONTROL-ETL'

08:57:41.446 | INFO    | Flow run 'mellow-cuttlefish' - View at https://app.prefect.cloud/account/645e4433-1321-4cfd-9a35-a54cbed1b6d6/workspace/e1e6136c-4435-4483-bb86-c454bd7494a4/runs/flow-run/06ac7a15-4f68-77d4-8000-4dd3c564ccf1

08:57:41.812 | INFO    | Task run 'extract_all_task-bdb' - Verificando y reutilizando Bronze local; sin descargar datos...

08:57:52.997 | INFO    | Task run 'extract_all_task-bdb' - Finished in state Completed()

08:57:53.031 | INFO    | Task run 'transform_task-c88' - Transformando procesos a Parquet...

...

10:34:42.810 | INFO    | Task run 'postgres_task-410' - gold.contratos_control: 1/1 archivos; 4,353,720 filas preparadas

10:36:34.915 | INFO    | Task run 'postgres_task-410' - Finished in state Completed()

10:36:36.182 | INFO    | Flow run 'mellow-cuttlefish' - Informe ETL: C:\Users\luna3\OneDrive\Desktop\secop_pipeline_equipo\notebooks\07_informe_ejecucion_etl.ipynb; histórico: C:\Users\luna3\OneDrive\Desktop\secop_pipeline_equipo\notebooks\informes_etl\20261008T135741-19e4f5d1.ipynb

10:36:37.468 | INFO    | Flow run 'mellow-cuttlefish' - Finished in state Completed()

[{'schema': 'secop', 'table': 'procesos', 'rows': 9070383, 'files': 117},
 {'schema': 'secop', 'table': 'contratos', 'rows': 5642731, 'files': 117},
 {'schema': 'secop', 'table': 'proveedores', 'rows': 1620376, 'files': 33},
 {'schema': 'secop', 'table': 'grupos', 'rows': 712905, 'files': 15},
 {'schema': 'gold', 'table': 'contratos_control', 'rows': 4353720, 'files': 1}]

## Cómo elegir la próxima ejecución

Cambie únicamente `MODO` en la celda anterior. Para recuperar la carga actual, mantenga `"solo_postgres"`. Para recalcular con datos locales, elija `"reprocesar_local"`. Para incorporar datos nuevos, elija `"actualizar_todo"`.

Los modos de flujo completo generan su informe de ejecución; la recuperación directa devuelve el resumen de carga en `resultado`. Las dos configuraciones Dask se ejecutan al construir Gold; no se ejecutan en `solo_postgres`.
